# Week 7 — Export Validation

Final validation of the ETL pipeline output.
68 automated checks + manual spot-checks.

## Setup (run pipeline)

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from src.pipeline import Pipeline

pipeline = Pipeline()
pipeline.run_bronze()
pipeline.run_silver()
pipeline.run_gold()

## Automated Validation (68 checks)

In [ ]:
from src.validation import OutputValidator

validator = OutputValidator("../../data/gold")
report = validator.run()

# Print full report
for c in report.checks:
    status = "PASS" if c.passed else "FAIL"
    print(f"[{status}] {c.name} -- {c.message}")

print(f"\nTotal: {len(report.checks)} checks -- {report.passed} passed, {report.failed} failed")

## Output File Overview

In [ ]:
from pathlib import Path

gold = Path("../../data/gold")
print(f"{'File':<30s}  {'Rows':>8s}  {'Cols':>5s}  {'Size KB':>8s}")
print("-" * 60)
for f in sorted(gold.glob("*.csv")):
    df = pd.read_csv(f, low_memory=False)
    print(f"{f.name:<30s}  {len(df):>8,}  {len(df.columns):>5}  {f.stat().st_size/1024:>8.1f}")

## Spot Checks

Compare raw source counts with processed outputs.

In [ ]:
# Spot-check: compare raw athlete count with dim_athlete
from src.utils import DataLoader
loader = DataLoader("../../data/raw")
raw_cert = loader.load_certifications()
empty_rows = raw_cert.isnull().all(axis=1).sum()
print(f"Raw certifications: {len(raw_cert):,} rows")
print(f"Empty rows: {empty_rows}")
print(f"Expected dim_athlete: {len(raw_cert) - empty_rows:,}")
print(f"Actual dim_athlete: {len(pipeline.dim_athlete):,}")
assert len(pipeline.dim_athlete) == len(raw_cert) - empty_rows

# Spot-check: raw results total vs silver deduped
raw_results = loader.load_all_results()
print(f"\nRaw results (all years): {len(raw_results):,}")
print(f"Silver (deduped): {len(pipeline.silver_results):,}")
print(f"Gold fact_results: {len(pipeline.fact_results):,}")

# Medal sanity: rank 1-3 should have medals
fr = pipeline.fact_results
medals = fr[fr["medal"].notna()]
print(f"\nMedals: {len(medals):,} ({fr['medal'].value_counts().to_dict()})")
no_medal = fr[(fr["rank"].notna()) & (fr["rank"] <= 3) & (fr["medal"].isna())]
print(f"Rank 1-3 without medal: {len(no_medal)} (should be 0)")

## Disqualification Rate Analysis (20% Rule)

In [ ]:
# 20% Rule: disqualification rate analysis
fr = pipeline.fact_results
total = len(fr)
dq = fr["is_disqualified"].sum()
print(f"Total results: {total:,}")
print(f"Disqualified: {dq:,} ({dq/total*100:.1f}%)")

# DQ rate by sport
sport_lookup = dict(zip(pipeline.dim_sport["sport_key"], pipeline.dim_sport["sport_name"]))
dq_by_sport = fr.groupby("sport_key").agg(
    total=("result_key", "count"),
    dq=("is_disqualified", "sum"),
).reset_index()
dq_by_sport["sport"] = dq_by_sport["sport_key"].map(sport_lookup)
dq_by_sport["dq_pct"] = (dq_by_sport["dq"] / dq_by_sport["total"] * 100).round(1)
print("\nDQ rate by sport:")
print(dq_by_sport[["sport", "total", "dq", "dq_pct"]].sort_values("dq_pct", ascending=False).to_string(index=False))

## Conclusion

All 68 validation checks pass. The ETL pipeline is complete:
- Bronze: 3 tables extracted
- Silver: 3 tables cleaned and deduped
- Gold: 5 dimensions + 2 facts in `data/gold/`
- 100% FK integrity across all relationships

Ready for Power BI import.